In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/SafeCue Audio Recordings"

print("Folder exists:", os.path.exists(BASE_DIR))
print("\nContents:")

for item in os.listdir(BASE_DIR):
    print("📁", item)


Folder exists: True

Contents:
📁 Siren
📁 Fire Alarm
📁 Smoke Alarm
📁 Screaming
📁 Background Noise
📁 _metadata


In [ ]:
!pip install -q yt-dlp
!apt-get -qq update
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.6 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os

METADATA_DIR = os.path.join(BASE_DIR, "_metadata")
os.makedirs(METADATA_DIR, exist_ok=True)

print("Metadata folder:", METADATA_DIR)


Metadata folder: /content/drive/MyDrive/SafeCue Audio Recordings/_metadata


In [ ]:
import pandas as pd

metadata_url = "https://research.google.com/audioset/download.html"

print("AudioSet metadata page:")
print(metadata_url)

AudioSet metadata page:
https://research.google.com/audioset/download.html


In [ ]:
!wget -q --show-progress \
"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/balanced_train_segments.csv" \
-o /dev/null

In [ ]:
import os

print(os.listdir(METADATA_DIR))

['unbalanced_train_segments.csv', 'siren_download_results.csv', 'fire_alarm_download_results.csv', 'smoke_alarm_download_results.csv', 'screaming_download_results.csv', 'balanced_train_segments.csv', 'ontology.json']


In [ ]:

import requests

url = "https://research.google.com/audioset/download.html"
response = requests.get(url)

print("Status code:", response.status_code)
print("Page downloaded:", len(response.text) > 0)


Status code: 200
Page downloaded: True


In [ ]:
import requests

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/balanced_train_segments.csv"

output_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

response = requests.get(url)

with open(output_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", output_path)
print("File size:", os.path.getsize(output_path), "bytes")

Downloaded: /content/drive/MyDrive/SafeCue Audio Recordings/_metadata/balanced_train_segments.csv
File size: 1211931 bytes


In [ ]:

csv_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

with open(csv_path, "r", encoding="utf-8") as f:
    for i in range(8):
        print(f"LINE {i+1}:", repr(f.readline()))


LINE 1: '# Segments csv created Sun Mar  5 10:54:31 2017\n'
LINE 2: '# num_ytids=22160, num_segs=22160, num_unique_labels=527, num_positive_labels=52882\n'
LINE 3: '# YTID, start_seconds, end_seconds, positive_labels\n'
LINE 4: '--PJHxphWEs, 30.000, 40.000, "/m/09x0r,/t/dd00088"\n'
LINE 5: '--ZhevVpy1s, 50.000, 60.000, "/m/012xff"\n'
LINE 6: '--aE2O5G5WE, 0.000, 10.000, "/m/03fwl,/m/04rlf,/m/09x0r"\n'
LINE 7: '--aO5cdqSAg, 30.000, 40.000, "/t/dd00003,/t/dd00005"\n'
LINE 8: '--aaILOrkII, 200.000, 210.000, "/m/032s66,/m/073cg4"\n'


In [ ]:
import csv
import pandas as pd
import os

csv_path = os.path.join(METADATA_DIR, "balanced_train_segments.csv")

rows = []

with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if row and not row[0].startswith("#"):
            rows.append(row)

df = pd.DataFrame(
    rows,
    columns=["YTID", "start_seconds", "end_seconds", "positive_labels"]
)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())


Shape: (22160, 4)

First 5 rows:


,YTID,start_seconds,end_seconds,positive_labels
0,--PJHxphWEs,30.000,40.000,"/m/09x0r,/t/dd00088"
1,--ZhevVpy1s,50.000,60.000,/m/012xff
2,--aE2O5G5WE,0.000,10.000,"/m/03fwl,/m/04rlf,/m/09x0r"
3,--aO5cdqSAg,30.000,40.000,"/t/dd00003,/t/dd00005"
4,--aaILOrkII,200.000,210.000,"/m/032s66,/m/073cg4"


In [ ]:
import requests
import json
import os

ontology_url = "https://raw.githubusercontent.com/audioset/ontology/master/ontology.json"
ontology_path = os.path.join(METADATA_DIR, "ontology.json")

response = requests.get(ontology_url)
response.raise_for_status()

with open(ontology_path, "wb") as f:
    f.write(response.content)

print("Ontology downloaded successfully!")
print("File size:", os.path.getsize(ontology_path), "bytes")

Ontology downloaded successfully!
File size: 342780 bytes


In [ ]:
import json

with open(ontology_path, "r", encoding="utf-8") as f:
    ontology = json.load(f)

keyword = "Outside, urban or manmade"

print(f"\n=== {keyword.upper()} ===")

matches = []

for item in ontology:
    name = item.get("name", "")

    if keyword.lower() in name.lower():
        matches.append((item.get("id"), name))

for audio_id, name in matches:
    print(audio_id, "→", name)


=== OUTSIDE, URBAN OR MANMADE ===
/t/dd00128 → Outside, urban or manmade


In [ ]:
background_noise = "/t/dd00128"
count = df["positive_labels"].apply(lambda x: background_noise in x.split(",")
    ).sum()
print(f"Background Noise: {count} clips")

Background Noise: 256 clips


In [ ]:
import requests
import os

url = "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/unbalanced_train_segments.csv"

output_path = os.path.join(METADATA_DIR, "unbalanced_train_segments.csv")

response = requests.get(url)
response.raise_for_status()

with open(output_path, "wb") as f:
    f.write(response.content)

print("Downloaded successfully!")
print("File size:", os.path.getsize(output_path), "bytes")

Downloaded successfully!
File size: 101468408 bytes


In [ ]:
import csv
from collections import Counter

unbalanced_path = os.path.join(
    METADATA_DIR,
    "unbalanced_train_segments.csv"
)

background_noise_id = "/t/dd00128"

counts = 0

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        if background_noise_id in labels:
          counts += 1

print("AudioSet unbalanced training counts:")
print("Background Noise in balanced dataset:", counts, "clips")

AudioSet unbalanced training counts:
Background Noise in balanced dataset: 33861 clips


In [ ]:
background_noise_id =  "/t/dd00128"
background_noise_rows = df[df["positive_labels"].apply(lambda x: background_noise_id in x.split(","))].reset_index(drop = True)
print("Background Noise Clips Available: ", len(background_noise_rows))
print("\nFirst Background Noise Clip: ")
display(background_noise_rows.head(1))

Background Noise Clips Available:  256

First Background Noise Clip: 


,YTID,start_seconds,end_seconds,positive_labels
0,-LBl0UeKNyU,430.000,440.000,"/m/07p6mqd,/m/09x0r,/t/dd00125,/t/dd00128"


In [ ]:
import subprocess
import os
import time

# Number of clips to test
NUM_TEST = 10

# Use the first 10 Background Noise clips
test_rows = background_noise_rows.head(NUM_TEST)

background_noise_folder = os.path.join(BASE_DIR, "Background Noise")
os.makedirs(background_noise_folder, exist_ok=True)

success = 0
failed = 0

for i, row in test_rows.iterrows():
    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        background_noise_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # Skip if already downloaded
    if os.path.exists(output_file):
        print(f"Already exists: {ytid}")
        success += 1
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(f"Downloading {success + failed + 1}/{NUM_TEST}: {ytid}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        success += 1
        print("  ✅ Success")
    else:
        failed += 1
        print("  ❌ Failed")

    time.sleep(1)

print("\n====================")
print("DOWNLOAD TEST COMPLETE")
print("====================")
print("Successful:", success)
print("Failed:", failed)

  ❌ Failed
Already exists: -RRi3QktLdc
Already exists: -WmL01c-4ZE
Already exists: -mNbbwgUZtQ
Already exists: 0gcJJ_M3gP0
Already exists: 0jaaX-we-8g
Already exists: 19DRe0cdeXQ
Already exists: 1ObaLGDunXY
Already exists: 1d0fsHq1qks
Already exists: 1qAvaKlkJis

DOWNLOAD TEST COMPLETE
Successful: 9
Failed: 1


In [ ]:
background_noise_candidates = background_noise_rows.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print(
    "Total Background Noise candidates:",
    len(background_noise_candidates)
)

print("\nFirst 10 candidates:")
display(background_noise_candidates.head(10))

Total Background Noise candidates: 256

First 10 candidates:


,YTID,start_seconds,end_seconds,positive_labels
0,-LBl0UeKNyU,430.000,440.000,"/m/07p6mqd,/m/09x0r,/t/dd00125,/t/dd00128"
1,-RRi3QktLdc,500.000,510.000,"/m/09x0r,/m/0ytgt,/t/dd00013,/t/dd00128"
2,-WmL01c-4ZE,20.000,30.000,"/m/07rjzl8,/m/09x0r,/t/dd00128"
3,-mNbbwgUZtQ,140.000,150.000,"/m/02mk9,/m/03p19w,/m/09x0r,/t/dd00128"
4,0gcJJ_M3gP0,30.000,40.000,"/m/06h7j,/m/09x0r,/t/dd00128"
5,0jaaX-we-8g,0.000,10.000,"/m/07qfr4h,/m/09x0r,/t/dd00128"
6,19DRe0cdeXQ,20.000,30.000,"/m/028v0c,/m/07r4wb8,/t/dd00128"
7,1ObaLGDunXY,50.000,60.000,"/m/04rlf,/m/09x0r,/t/dd00127,/t/dd00128"
8,1d0fsHq1qks,160.000,170.000,"/m/04rlf,/m/07p6mqd,/m/09x0r,/t/dd00128"
9,1qAvaKlkJis,30.000,40.000,"/m/07qn4z3,/t/dd00128"


In [ ]:
import csv
import pandas as pd
import os

unbalanced_path = os.path.join(
    METADATA_DIR,
    "unbalanced_train_segments.csv"
)

background_noise_id = "/t/dd00128"

background_noise_unbalanced_rows = []

with open(unbalanced_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f, skipinitialspace=True)

    for row in reader:
        if not row or row[0].startswith("#"):
            continue

        labels = row[3].split(",")

        if background_noise_id in labels:
            background_noise_unbalanced_rows.append(row)

background_noise_unbalanced = pd.DataFrame(
    background_noise_unbalanced_rows,
    columns=[
        "YTID",
        "start_seconds",
        "end_seconds",
        "positive_labels"
    ]
)

# Remove duplicate YouTube videos
background_noise_unbalanced = background_noise_unbalanced.drop_duplicates(
    subset=["YTID"]
).reset_index(drop=True)

print(
    "Unique Background Noise candidates in unbalanced set:",
    len(background_noise_unbalanced)
)

display(background_noise_unbalanced.head(10))


Unique Background Noise candidates in unbalanced set: 33861


,YTID,start_seconds,end_seconds,positive_labels
0,--7VmHMn_b4,30.000,40.000,"/m/04rlf,/m/09x0r,/t/dd00128"
1,--K2ZlsdQ4o,30.000,40.000,"/m/07yv9,/t/dd00128"
2,--KYALrhHy4,30.000,40.000,"/m/03qtwd,/m/04rlf,/m/09x0r,/t/dd00128"
3,--KxycEjQtU,60.000,70.000,"/m/09x0r,/t/dd00128"
4,--OqV2-d2E4,280.000,290.000,"/m/07yv9,/m/09x0r,/m/0k4j,/t/dd00128"
5,--PMnfMhn1A,30.000,40.000,"/m/03qtwd,/t/dd00128"
6,--_VuZzBX74,190.000,200.000,"/m/02lkt,/m/04rlf,/t/dd00128"
7,--du6cuRWZs,170.000,180.000,"/m/04rlf,/t/dd00128"
8,--gKfbzT1VY,20.000,30.000,"/m/012ndj,/m/03j1ly,/m/07r04,/m/07yv9,/t/dd00128"
9,--k9htntelY,30.000,40.000,"/m/04rlf,/m/09x0r,/t/dd00128"


In [ ]:
BACKGROUND_NOISE_TARGET = 300

background_noise_download_candidates = background_noise_unbalanced.sample(
    n=BACKGROUND_NOISE_TARGET,
    random_state=42
).reset_index(drop=True)

print("Background Noise Candidates selected:", len(background_noise_download_candidates))

display(background_noise_download_candidates.head(10))


Background Noise Candidates selected: 400


,YTID,start_seconds,end_seconds,positive_labels
0,hxoO4q8EZ0Q,60.000,70.000,"/m/09x0r,/t/dd00128"
1,L2u6Z70YCeg,100.000,110.000,"/m/09x0r,/t/dd00128"
2,u8yRw5XZbFY,140.000,150.000,"/m/012f08,/m/07yv9,/m/0k4j,/t/dd00128"
3,uWNsmuVYJRE,400.000,410.000,"/m/07r04,/m/07yv9,/m/09x0r,/t/dd00128"
4,iWz1YklxRZE,30.000,40.000,"/m/06h7j,/m/09x0r,/t/dd00128"
5,OhGg9w1n5MU,10.000,20.000,"/m/03k3r,/m/04rlf,/m/0jbk,/t/dd00128"
6,A2hfPy_apIU,5.000,15.000,"/m/0912c9,/t/dd00128"
7,jp3AId2qH9E,90.000,100.000,"/m/04rlf,/m/09x0r,/t/dd00128"
8,5CmhBfwuSv8,30.000,40.000,/t/dd00128
9,a-zpKuDFBUg,30.000,40.000,"/m/04_sv,/m/07yv9,/t/dd00128"


In [ ]:
import subprocess
import os
import time
import pandas as pd

background_noise_folder = os.path.join(BASE_DIR, "Background Noise")
os.makedirs(background_noise_folder, exist_ok=True)

results = []

for i, row in background_noise_download_candidates.iterrows():

    ytid = row["YTID"]
    start = row["start_seconds"]
    end = row["end_seconds"]

    output_file = os.path.join(
        background_noise_folder,
        f"{ytid}_{start}_{end}.wav"
    )

    # If already downloaded, don't download again
    if os.path.exists(output_file):
        print(
            f"{i+1}/{len(background_noise_download_candidates)} "
            f"✅ Already exists: {ytid}"
        )

        results.append({
            "YTID": ytid,
            "start_seconds": start,
            "end_seconds": end,
            "label": "Background Noise",
            "filename": output_file,
            "status": "already_exists"
        })
        continue

    url = f"https://www.youtube.com/watch?v={ytid}"

    command = [
        "yt-dlp",
        "--no-playlist",
        "--quiet",
        "--no-warnings",
        "--extract-audio",
        "--audio-format", "wav",
        "--download-sections", f"*{start}-{end}",
        "-o", output_file,
        url
    ]

    print(
        f"{i+1}/{len(background_noise_download_candidates)} "
        f"Downloading: {ytid}"
    )

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    if result.returncode == 0 and os.path.exists(output_file):
        status = "success"
        print("   ✅ Success")
    else:
        status = "failed"
        print("   ❌ Failed")
        print("   ERROR:", result.stderr)
    results.append({
        "YTID": ytid,
        "start_seconds": start,
        "end_seconds": end,
        "label": "Background Noise",
        "filename": output_file if status == "success" else "",
        "status": status
    })

    time.sleep(1)

# Save download results
download_results = pd.DataFrame(results)

results_path = os.path.join(
    METADATA_DIR,
    "background_noise_download_results.csv"
)

download_results.to_csv(results_path, index=False)

print("\n====================================")
print("BACKGROUND NOISE DOWNLOAD COMPLETE")
print("====================================")
print(
    "Successful:",
    (download_results["status"] == "success").sum()
)
print(
    "Already existed:",
    (download_results["status"] == "already_exists").sum()
)
print(
    "Failed:",
    (download_results["status"] == "failed").sum()
)
print("Results saved to:", results_path)

1/400 ✅ Already exists: hxoO4q8EZ0Q
2/400 ✅ Already exists: L2u6Z70YCeg
3/400 ✅ Already exists: u8yRw5XZbFY
4/400 ✅ Already exists: uWNsmuVYJRE
5/400 ✅ Already exists: iWz1YklxRZE
6/400 ✅ Already exists: OhGg9w1n5MU
7/400 ✅ Already exists: A2hfPy_apIU
8/400 ✅ Already exists: jp3AId2qH9E
9/400 ✅ Already exists: 5CmhBfwuSv8
10/400 ✅ Already exists: a-zpKuDFBUg
11/400 ✅ Already exists: rD2sMlMM2TU
12/400 ✅ Already exists: xU5_flIx-dA
13/400 ✅ Already exists: nCGKsjIJbiE
14/400 Downloading: YhJ1NaLBj9U
   ❌ Failed
   ERROR: ERROR: [youtube] YhJ1NaLBj9U: Video unavailable

15/400 ✅ Already exists: 0jS8QrZDNtY
16/400 ✅ Already exists: dClkN6MtQcE
17/400 ✅ Already exists: II7LEtbM4IU
18/400 ✅ Already exists: 5fyzKTSL20w
19/400 ✅ Already exists: EyI1dCPaJPg
20/400 ✅ Already exists: Hz9XowPzXxI
21/400 Downloading: cSrv7Ed-Au4
   ❌ Failed
   ERROR: ERROR: [youtube] cSrv7Ed-Au4: This video is not available

22/400 ✅ Already exists: 710INRXyTus
23/400 ✅ Already exists: l5XFK0ChW6A
24/400 ✅ Already

KeyboardInterrupt: 